# Spark on Apache Hadoop YARN: Architecture and Resource Allocation

**Module**: 12_advanced_topics  
**Notebook**: 01_spark_on_yarn.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark runs on Hadoop YARN: ResourceManager, NodeManager, ApplicationMaster, and yarn-client vs yarn-cluster modes.
- **Why does this topic exist?**: YARN is the dominant cluster manager in enterprise on-premises Hadoop data platforms.
- **What problem does it solve?**: Configuring YARN memory overhead and queue allocations.
- **Where is it used in real-world Data Engineering?**: Enterprise banking and telecom Hadoop clusters.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Cluster architecture, resource managers, deployment modes |
| **Required depth** | Deep working knowledge: can explain YARN components and Spark's resource request flow |
| **Data Engineer** | Submit and diagnose Spark jobs on a YARN cluster. |
| **Senior Data Engineer** | Tune queues, containers, and deployment modes for shared clusters. |
| **Spark specialist** | Reason about ApplicationMaster behavior, dynamic allocation, and YARN scheduling. |
| **Next topic** | `12_advanced_topics/02_spark_on_kubernetes.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

YARN is a cluster resource manager that allocates containers to applications. Spark on YARN runs a driver
(either on the submitting machine or inside an ApplicationMaster container) and requests executor containers from
the ResourceManager via the ApplicationMaster. Understanding the control plane explains queue waits, container
sizing, and why a job can be accepted but never start.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **ResourceManager** | Cluster resource broker | Allocates containers across queues |
| **NodeManager** | Per-node agent | Launches and monitors containers |
| **ApplicationMaster** | Per-application coordinator | Requests containers and monitors the Spark application |
| **Container** | Resource allocation | Cores and memory granted to a driver or executor |
| **Queue** | Scheduling pool | Capacity and fair-scheduling policies |
| **Deploy mode** | Driver location | Client mode on the submit host; cluster mode in the cluster |

### How it works, step by step

1. The client submits an application to the ResourceManager.
2. The ResourceManager allocates an ApplicationMaster container.
3. The ApplicationMaster starts the driver (cluster mode) or connects to the client driver.
4. The driver requests executor containers from the ApplicationMaster.
5. The ApplicationMaster asks the ResourceManager for resources and launches executors on NodeManagers.
6. Executors register with the driver and the job begins.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Spark on YARN Architecture
```
YARN Cluster:
  - ResourceManager (RM): Central cluster arbiter.
  - NodeManager (NM): Runs on each worker node, launching containers.
Application Submission:
  1. Client submits job to RM.
  2. RM allocates Container 0 for Spark ApplicationMaster (AM).
  3. In yarn-cluster mode, the AM hosts the Spark Driver!
  4. In yarn-client mode, Driver runs on client; AM only requests executors.
  5. AM requests executor containers across NodeManagers.
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("YARN_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("Cluster Manager in our Lab: Spark Standalone (spark://spark-master:7077)")
print("YARN submission equivalent command:")
print("spark-submit --master yarn --deploy-mode cluster --num-executors 10 --executor-memory 4g my_app.py")

---

## Experiment: Comparing Standalone vs YARN

- **1. What are we testing?**: Contrast Standalone Master with YARN ResourceManager.
- **2. What do we expect?**: Standalone is dedicated to Spark; YARN manages multi-tenant workloads (Hive, Flink, Spark, MapReduce).

In [ ]:
print('YARN provides multi-tenant queue scheduling (CapacityScheduler, FairScheduler).')

**3. What happened?**: Explained YARN architectural distinctions.  
**4. Why did it happen?**: YARN is a generic resource manager, not Spark-specific.  
**5. Practical Engineering Lesson**: Use Standalone for dedicated Spark clusters; use YARN for multi-tenant Hadoop clusters.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Shared cluster queue contention

- **Industry**: Enterprise data platform
- **Scenario**: A finance Spark job waits hours in a YARN queue while other jobs run.
- **Problem**: The queue capacity and scheduling policy determine when the application gets containers.
- **Why this concept is relevant**: Understanding queue placement and resource requests lets the team choose the right queue and size.
- **Implementation idea**: Inspect queue metrics, submit to the appropriate queue, and compare requested versus available resources.
- **Expected result**: A job that starts within its SLA window.
- **Alternative**: Reserve a dedicated queue.
- **Trade-offs**: A dedicated queue guarantees capacity but may leave resources idle outside peak periods.

### Use Case 2: Container memory mismatch

- **Industry**: Telecom
- **Scenario**: A Spark job on YARN is killed for exceeding container memory.
- **Problem**: The executor heap plus overhead exceeds the container size requested from YARN.
- **Why this concept is relevant**: Aligning `spark.executor.memory`, overhead, and the YARN container size prevents the kill.
- **Implementation idea**: Check the YARN container diagnostics and adjust the memory settings together.
- **Expected result**: Stable executors and fewer retries.
- **Alternative**: Increase the container size.
- **Trade-offs**: A larger container may wait longer for allocation and waste capacity if the job is small.

---

## Business Value

- Better utilization of shared cluster capacity.
- Fewer queue-related SLA misses.
- Stable container sizing and fewer kills.
- Clear escalation paths for resource issues.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Check the queue and scheduling policy before blaming Spark.
- Size the container as heap plus overhead.
- Use cluster mode for production submissions.
- Monitor ApplicationMaster and NodeManager logs.
- Understand that dynamic allocation interacts with queue capacity.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Trace a YARN submission

**Steps**

1. Submit a small Spark job in cluster mode to a YARN queue.
2. Open the ResourceManager UI and find the application and ApplicationMaster.
3. Inspect the container allocation and executor registration.

**What to observe**: The ApplicationMaster requests containers and the driver receives executor registrations.

**Healthy result**: The job starts with the expected number of executors.

**Problematic result**: The application is accepted but remains pending due to queue capacity.

**Engineering takeaway**: YARN scheduling is a separate layer from Spark's own scheduling.

---

## Trade-offs and Alternatives

### Advantages

- Mature resource management
- Multi-tenant queues
- Integration with Hadoop ecosystem

### Limitations

- Operational complexity
- Queue waits
- Container sizing pitfalls

### When to use it

- On-prem Hadoop clusters
- Shared enterprise platforms
- Batch workloads

### When not to use it

- Small ad-hoc jobs on a heavily queued cluster
- Ignoring overhead
- Client mode for production

### Alternatives inside Spark

- ApplicationMaster
- Executor containers
- Dynamic allocation

### Alternatives outside Spark

- Kubernetes
- Standalone mode
- Managed Spark services

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): YARN Deploy Modes
**Problem Statement**: What is the difference between yarn-client and yarn-cluster?  
**Input / Context**: Modes

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): YARN Memory Overhead Config
**Problem Statement**: What is spark.yarn.executor.memoryOverhead?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): CapacityScheduler Queues
**Problem Statement**: How to specify a YARN queue in spark-submit.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The job that never starts

**Scenario**: A Spark application is accepted by YARN but stays in ACCEPTED state for hours.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check queue capacity and the number of pending applications.
2. Compare requested resources with the queue's available resources.
3. Inspect the ResourceManager scheduler logs.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (YARN Deploy Modes):
```python
print('In yarn-client, Driver runs on client machine; in yarn-cluster, Driver runs inside YARN ApplicationMaster container.')
```

#### Solution 2 (YARN Memory Overhead Config):
```python
print('Off-heap container buffer for YARN (default max(384MB, 10% of heap)).')
```

#### Solution 3 (CapacityScheduler Queues):
```python
print('--queue production_etl')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **YARN AM**: ApplicationMaster negotiates containers.
- **Deploy Modes**: yarn-client for dev/Jupyter; yarn-cluster for production.

### Essential Technical Terms
- `ResourceManager`: Central YARN scheduler.
- `ApplicationMaster`: Per-application coordinator in YARN.

### Common Engineering Mistakes
- **yarn-client in production cron jobs**: If client terminal closes, job terminates.

### Interview Questions on This Topic
- **Q: Where does the Driver run in yarn-cluster mode?**
  - *A*: Inside the YARN ApplicationMaster container on a worker node in the cluster.


### More Interview Questions

- **Q: What is the role of the ApplicationMaster in Spark on YARN?**
  - *A*: It coordinates the application: requests containers from the ResourceManager, launches executors, and monitors the application; in cluster mode it also hosts the driver.
- **Q: Why does container memory need to exceed executor heap?**
  - *A*: The container includes JVM overhead, native memory, and other process costs, so heap alone is not enough and the container can be killed if it exceeds its limit.

### What to Study Next
Open 02_spark_on_kubernetes.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')